# 02 — Experimental evaluation: top-down vs top-down + bottom-up

Questo notebook **non genera goal** e **non esegue il feedback loop**. Legge due insiemi di output già persistiti:

- `output/top_down/`: baseline top-down;
- `output/top_down_bottom_up/`: lo stesso stato iniziale dopo il ciclo bottom-up.

Per ciascun dataset confronta separatamente HLG e LLG contro la ground truth usando la stessa metrica semantica BERT/Hungarian già presente nel notebook di valutazione precedente. Calcola quindi **Precision, Recall e F1** per:

1. top-down only;
2. top-down + bottom-up;
3. delta bottom-up − top-down.

Il risultato permette di verificare direttamente se il feedback bottom-up migliora o peggiora la baseline, senza mescolare generazione ed evaluation nello stesso notebook.

In [1]:
from pathlib import Path
import json
import os
import socket
import sys

os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")

import nltk
import numpy as np
import pandas as pd
import torch
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_BOTTOM_UP_DIR = OUTPUT_ROOT / "top_down_bottom_up"
EVALUATION_DIR = OUTPUT_ROOT / "evaluation" / "top_down_vs_bottom_up"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

from groundtruth import ALL_GROUNDTRUTHS, GROUNDTRUTH_BY_NAME

GROUNDTRUTHS = list(ALL_GROUNDTRUTHS)

print("Project root:", PROJECT_ROOT)
print("Top-down directory:", TOP_DOWN_DIR)
print("Top-down + bottom-up directory:", TOP_DOWN_BOTTOM_UP_DIR)
print("Evaluation output:", EVALUATION_DIR)

c:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement
Top-down directory: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down
Top-down + bottom-up directory: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down_bottom_up
Evaluation output: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\top_down_vs_bottom_up


## 1. Caricamento e pairing degli output

Il pairing viene fatto sul campo `name` del JSON, non sulla posizione dei file. Un dataset entra nel confronto solo se sono disponibili sia la baseline sia l'output con feedback e se esiste la relativa ground truth.

In [2]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def index_json_files(directory: Path, pattern: str):
    result = {}
    for path in sorted(directory.glob(pattern)):
        payload = load_json(path)
        dataset_name = payload.get("name")
        if not dataset_name:
            print(f"Skipping file without dataset name: {path}")
            continue
        if dataset_name in result:
            raise RuntimeError(
                f"More than one file found for dataset {dataset_name!r} in {directory}. "
                "Remove stale duplicates or narrow the file pattern."
            )
        result[dataset_name] = (path, payload)
    return result


baseline_by_dataset = index_json_files(TOP_DOWN_DIR, "*_top_down.json")
combined_by_dataset = index_json_files(TOP_DOWN_BOTTOM_UP_DIR, "*_top_down_bottom_up.json")
combined_by_dataset = {
    name: entry
    for name, entry in combined_by_dataset.items()
    if not entry[1].get("bottomUpError")
    and entry[1].get("finalHighLevelGoals") is not None
    and entry[1].get("finalLowLevelGoals") is not None
}

common_datasets = sorted(
    set(baseline_by_dataset)
    & set(combined_by_dataset)
    & set(GROUNDTRUTH_BY_NAME)
)

if not common_datasets:
    raise RuntimeError(
        "No complete baseline/feedback pair found. Run the two execution notebooks first."
    )

print(f"Comparable datasets: {len(common_datasets)}")
for dataset_name in common_datasets:
    print(
        f"- {dataset_name}\n"
        f"    baseline: {baseline_by_dataset[dataset_name][0].name}\n"
        f"    feedback: {combined_by_dataset[dataset_name][0].name}"
    )

missing_feedback = sorted(set(baseline_by_dataset) - set(combined_by_dataset))
if missing_feedback:
    print("\nBaselines without a bottom-up result:", missing_feedback)

Comparable datasets: 8
- Assegno Unico Universale - SIA Project 24 25
    baseline: Assegno Unico Universale - SIA Project 24 25_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: Assegno Unico Universale - SIA Project 24 25_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json
- Ethical Purchasing Group - SIA Project 22 23
    baseline: Ethical Purchasing Group - SIA Project 22 23_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: Ethical Purchasing Group - SIA Project 22 23_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json
- Event Organization Portal - SIA Project 21 22
    baseline: Event Organization Portal - SIA Project 21 22_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: Event Organization Portal - SIA Project 21 22_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json
- Genome Nexus
    baseline: Genome Nexus_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: Genome Nexus_ZERO_SHOT_feedback_orig

## 2. Estrazione delle collezioni di goal

La baseline usa le liste `highLevelGoals` e `lowLevelGoals`. L'output con feedback usa invece le collezioni finali restituite dall'orchestrator (`finalHighLevelGoals`, `finalLowLevelGoals`).

In [3]:
def goal_text(item) -> str:
    if item is None:
        return ""
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        name = item.get("name")
        if isinstance(name, str) and name.strip():
            return name.strip()
        description = item.get("description")
        if isinstance(description, str):
            return description.strip()
    return str(item).strip()


def extract_hlg(container):
    if container is None:
        return []
    items = container.get("goals", []) if isinstance(container, dict) else container
    return [text for item in items if (text := goal_text(item))]


def extract_llg(container):
    if container is None:
        return []
    items = container.get("low_level_goals", []) if isinstance(container, dict) else container
    return [text for item in items if (text := goal_text(item))]


def extract_cycle_status(payload):
    cycle = payload.get("globalGoalCycle") or {}
    return {
        "converged": bool(cycle.get("converged")),
        "stop_reason": cycle.get("stop_reason"),
        "completed_iterations": cycle.get("completed_iterations"),
    }


states = {}
for dataset_name in common_datasets:
    _, baseline = baseline_by_dataset[dataset_name]
    _, combined = combined_by_dataset[dataset_name]
    gt = GROUNDTRUTH_BY_NAME[dataset_name]

    states[dataset_name] = {
        "reference_hlg": [str(x).strip() for x in gt["highLevelGoals"] if str(x).strip()],
        "reference_llg": [str(x).strip() for x in gt["lowLevelGoals"] if str(x).strip()],
        "top_down_hlg": extract_hlg(baseline.get("structuredHighLevelGoals") or baseline.get("highLevelGoals")),
        "top_down_llg": extract_llg(baseline.get("structuredLowLevelGoals") or baseline.get("lowLevelGoals")),
        "feedback_hlg": extract_hlg(combined.get("finalHighLevelGoals")),
        "feedback_llg": extract_llg(combined.get("finalLowLevelGoals")),
        "cycle_status": extract_cycle_status(combined),
    }

for dataset_name, s in states.items():
    print(
        f"{dataset_name}: "
        f"GT HLG/LLG={len(s['reference_hlg'])}/{len(s['reference_llg'])} | "
        f"top-down={len(s['top_down_hlg'])}/{len(s['top_down_llg'])} | "
        f"+bottom-up={len(s['feedback_hlg'])}/{len(s['feedback_llg'])} | "
        f"converged={s['cycle_status']['converged']}"
    )

Assegno Unico Universale - SIA Project 24 25: GT HLG/LLG=4/35 | top-down=15/43 | +bottom-up=15/44 | converged=True
Ethical Purchasing Group - SIA Project 22 23: GT HLG/LLG=5/30 | top-down=12/44 | +bottom-up=12/43 | converged=True
Event Organization Portal - SIA Project 21 22: GT HLG/LLG=3/30 | top-down=11/61 | +bottom-up=10/56 | converged=True
Genome Nexus: GT HLG/LLG=9/34 | top-down=10/31 | +bottom-up=8/25 | converged=True
Gestao Hospital: GT HLG/LLG=4/20 | top-down=10/33 | +bottom-up=6/20 | converged=False
La Reine Marlene - SIA Project 23 24: GT HLG/LLG=10/50 | top-down=24/86 | +bottom-up=19/70 | converged=False
London Ambulance Service: GT HLG/LLG=2/10 | top-down=10/30 | +bottom-up=8/24 | converged=True
SIA Project 25 26: GT HLG/LLG=11/37 | top-down=14/28 | +bottom-up=13/32 | converged=True


## 3. Preparazione della metrica BERT/Hungarian

La metrica è la stessa logica usata nel precedente notebook sperimentale:

- preprocessing NLTK;
- embedding `bert-base-uncased` con mean pooling;
- cosine similarity tra goal generati e goal di riferimento;
- matching 1-a-1 ottimo tramite algoritmo ungherese;
- soft Precision = similarità totale / numero di goal generati;
- soft Recall = similarità totale / numero di goal di riferimento;
- F1 armonica di Precision e Recall.

In [4]:
NLTK_RESOURCES = {
    "punkt": ("tokenizers/punkt",),
    "punkt_tab": ("tokenizers/punkt_tab",),
    "wordnet": ("corpora/wordnet", "corpora/wordnet.zip"),
    "omw-1.4": ("corpora/omw-1.4", "corpora/omw-1.4.zip"),
    "stopwords": ("corpora/stopwords",),
}


def _resource_exists(candidates):
    for candidate in candidates:
        try:
            nltk.data.find(candidate)
            return True
        except LookupError:
            pass
    return False


previous_timeout = socket.getdefaulttimeout()
socket.setdefaulttimeout(45)
try:
    for package, candidates in NLTK_RESOURCES.items():
        if not _resource_exists(candidates):
            print(f"Downloading missing NLTK resource: {package}")
            ok = nltk.download(package, quiet=False, raise_on_error=False)
            if not ok:
                raise RuntimeError(f"Could not download NLTK resource: {package}")
finally:
    socket.setdefaulttimeout(previous_timeout)

BERT_MODEL_NAME = "bert-base-uncased"
BERT_BATCH_SIZE = 16
SIMILARITY_THRESHOLD = 0.65

print(f"Loading {BERT_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModel.from_pretrained(BERT_MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

print("Device:", device)

Loading bert-base-uncased...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3234.89it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cpu


In [5]:
class PaperAlignedGoalEvaluator:
    def __init__(self, tokenizer, model, device, batch_size=16, preprocess=False, similarity_threshold=0.65):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.batch_size = batch_size
        self.preprocess = preprocess
        self.embedding_cache = {}
        self.similarity_threshold = similarity_threshold

    def preprocess_text(self, text):
        tokens = word_tokenize(text)
        filtered_tokens = [
            token for token in tokens
            if token not in stop_words and token.isalnum()
        ]
        stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]
        lemmatized_tokens = [lemmatizer.lemmatize(token) for token in stemmed_tokens]
        return " ".join(lemmatized_tokens)

    def _encode_missing(self, raw_texts):
        missing = [text for text in raw_texts if text not in self.embedding_cache]
        if not missing:
            return

        prepared = [
            self.preprocess_text(text) if self.preprocess else text
            for text in missing
        ]

        total_batches = (len(missing) + self.batch_size - 1) // self.batch_size
        for start in tqdm(
            range(0, len(missing), self.batch_size),
            total=total_batches,
            desc="BERT batches",
        ):
            raw_batch = missing[start:start + self.batch_size]
            text_batch = prepared[start:start + self.batch_size]

            encoded_input = self.tokenizer(
                text_batch,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)

            with torch.inference_mode():
                model_output = self.model(**encoded_input)

            token_embeddings = model_output.last_hidden_state
            attention_mask = encoded_input["attention_mask"]
            input_mask_expanded = (
                attention_mask.unsqueeze(-1)
                .expand(token_embeddings.size())
                .float()
            )

            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
            sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
            pooled = (sum_embeddings / sum_mask).detach().cpu()

            for text, embedding in zip(raw_batch, pooled):
                self.embedding_cache[text] = embedding

    def encode_texts(self, texts):
        self._encode_missing(texts)
        return torch.stack([self.embedding_cache[text] for text in texts])

    def compute_similarity(self, generated_goals, reference_goals):
        gen_embeddings = self.encode_texts(generated_goals)
        ref_embeddings = self.encode_texts(reference_goals)
        gen_norm = torch.nn.functional.normalize(gen_embeddings, p=2, dim=1)
        ref_norm = torch.nn.functional.normalize(ref_embeddings, p=2, dim=1)
        return torch.mm(gen_norm, ref_norm.transpose(0, 1)).numpy()

    def calculate_soft_f1(self, preds, refs):
        if not preds or not refs:
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "matched_goal_count": 0}

        sim_matrix = self.compute_similarity(preds, refs)
        row_ind, col_ind = linear_sum_assignment(-sim_matrix)
        matched_similarities = sim_matrix[row_ind, col_ind]
        accepted = matched_similarities[matched_similarities >= self.similarity_threshold]
        total_similarity = float(accepted.sum())

        precision = total_similarity / len(preds)
        recall = total_similarity / len(refs)
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "matched_goal_count": int(len(accepted)),
        }


evaluator = PaperAlignedGoalEvaluator(
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=BERT_BATCH_SIZE,
    preprocess=False,
    similarity_threshold=SIMILARITY_THRESHOLD,
)

## 4. Precision, Recall e F1 separati per baseline e feedback

Ogni riga usa la stessa ground truth e cambia solo lo stato generato. Questo è il confronto principale da usare per rispondere alla domanda sperimentale: **il bottom-up migliora la qualità dei goal rispetto alla stessa pipeline top-down senza feedback?**

In [6]:
metric_rows = []

for dataset_name, s in states.items():
    status = s["cycle_status"]
    for level, reference, top_down, feedback in (
        (
            "High-Level Goals",
            s["reference_hlg"],
            s["top_down_hlg"],
            s["feedback_hlg"],
        ),
        (
            "Low-Level Goals",
            s["reference_llg"],
            s["top_down_llg"],
            s["feedback_llg"],
        ),
    ):
        for method, generated in (
            ("top_down", top_down),
            ("top_down_plus_bottom_up", feedback),
        ):
            m = evaluator.calculate_soft_f1(generated, reference)
            metric_rows.append(
                {
                    "dataset": dataset_name,
                    "level": level,
                    "method": method,
                    "generated_goal_count": len(generated),
                    "reference_goal_count": len(reference),
                    "precision": m["precision"],
                    "recall": m["recall"],
                    "f1": m["f1"],
                    "matched_goal_count": m["matched_goal_count"],
                    "unmatched_generated_count": len(generated) - m["matched_goal_count"],
                    "cycle_converged": status["converged"],
                    "cycle_stop_reason": status["stop_reason"],
                    "cycle_completed_iterations": status["completed_iterations"],
                }
            )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(4))

BERT batches: 100%|██████████| 1/1 [00:00<00:00,  8.47it/s]


,dataset,level,method,generated_goal_count,reference_goal_count,precision,recall,f1,matched_goal_count,unmatched_generated_count,cycle_converged,cycle_stop_reason,cycle_completed_iterations
0,Assegno Unico Universale - SIA Project 24 25,High-Level Goals,top_down,15,4,0.2070,0.7764,0.3269,4,11,True,ALL_BRANCHES_CONFIRMED,2
1,Assegno Unico Universale - SIA Project 24 25,High-Level Goals,top_down_plus_bottom_up,15,4,0.2070,0.7764,0.3269,4,11,True,ALL_BRANCHES_CONFIRMED,2
2,Assegno Unico Universale - SIA Project 24 25,Low-Level Goals,top_down,43,35,0.6482,0.7963,0.7147,35,8,True,ALL_BRANCHES_CONFIRMED,2
3,Assegno Unico Universale - SIA Project 24 25,Low-Level Goals,top_down_plus_bottom_up,44,35,0.6341,0.7971,0.7063,35,9,True,ALL_BRANCHES_CONFIRMED,2
4,Ethical Purchasing Group - SIA Project 22 23,High-Level Goals,top_down,12,5,0.3545,0.8507,0.5004,5,7,True,ALL_BRANCHES_CONFIRMED,2
5,Ethical Purchasing Group - SIA Project 22 23,High-Level Goals,top_down_plus_bottom_up,12,5,0.3545,0.8507,0.5004,5,7,True,ALL_BRANCHES_CONFIRMED,2
6,Ethical Purchasing Group - SIA Project 22 23,Low-Level Goals,top_down,44,30,0.5135,0.7531,0.6107,29,15,True,ALL_BRANCHES_CONFIRMED,2
7,Ethical Purchasing Group - SIA Project 22 23,Low-Level Goals,top_down_plus_bottom_up,43,30,0.5254,0.7531,0.6190,29,14,True,ALL_BRANCHES_CONFIRMED,2
8,Event Organization Portal - SIA Project 21 22,High-Level Goals,top_down,11,3,0.2196,0.8051,0.3450,3,8,True,ALL_BRANCHES_CONFIRMED,2
9,Event Organization Portal - SIA Project 21 22,High-Level Goals,top_down_plus_bottom_up,10,3,0.2415,0.8051,0.3716,3,7,True,ALL_BRANCHES_CONFIRMED,2


## 5. Delta bottom-up − top-down

Un delta positivo indica miglioramento; un delta negativo indica peggioramento. Precision e Recall vanno interpretati insieme: ad esempio, aggiungere molti goal può aumentare Recall ma ridurre Precision.

In [7]:
baseline_df = (
    metrics_df[metrics_df["method"] == "top_down"]
    .set_index(["dataset", "level"])
)
feedback_df = (
    metrics_df[metrics_df["method"] == "top_down_plus_bottom_up"]
    .set_index(["dataset", "level"])
)

comparison_rows = []
for key in baseline_df.index.intersection(feedback_df.index):
    dataset_name, level = key
    base = baseline_df.loc[key]
    feedback = feedback_df.loc[key]
    comparison_rows.append(
        {
            "dataset": dataset_name,
            "level": level,
            "top_down_goal_count": int(base["generated_goal_count"]),
            "bottom_up_goal_count": int(feedback["generated_goal_count"]),
            "delta_goal_count": int(feedback["generated_goal_count"] - base["generated_goal_count"]),
            "top_down_precision": base["precision"],
            "bottom_up_precision": feedback["precision"],
            "delta_precision": feedback["precision"] - base["precision"],
            "top_down_recall": base["recall"],
            "bottom_up_recall": feedback["recall"],
            "delta_recall": feedback["recall"] - base["recall"],
            "top_down_f1": base["f1"],
            "bottom_up_f1": feedback["f1"],
            "delta_f1": feedback["f1"] - base["f1"],
            "cycle_converged": bool(feedback["cycle_converged"]),
            "cycle_stop_reason": feedback["cycle_stop_reason"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

print("\nRows where bottom-up improved F1:")
display(comparison_df[comparison_df["delta_f1"] > 0].sort_values("delta_f1", ascending=False).round(4))

print("\nRows where bottom-up worsened F1:")
display(comparison_df[comparison_df["delta_f1"] < 0].sort_values("delta_f1").round(4))

,dataset,level,top_down_goal_count,bottom_up_goal_count,delta_goal_count,top_down_precision,bottom_up_precision,delta_precision,top_down_recall,bottom_up_recall,delta_recall,top_down_f1,bottom_up_f1,delta_f1,cycle_converged,cycle_stop_reason
0,Assegno Unico Universale - SIA Project 24 25,High-Level Goals,15,15,0,0.2070,0.2070,0.0000,0.7764,0.7764,0.0000,0.3269,0.3269,0.0000,True,ALL_BRANCHES_CONFIRMED
1,Assegno Unico Universale - SIA Project 24 25,Low-Level Goals,43,44,1,0.6482,0.6341,-0.0141,0.7963,0.7971,0.0008,0.7147,0.7063,-0.0083,True,ALL_BRANCHES_CONFIRMED
2,Ethical Purchasing Group - SIA Project 22 23,High-Level Goals,12,12,0,0.3545,0.3545,0.0000,0.8507,0.8507,0.0000,0.5004,0.5004,0.0000,True,ALL_BRANCHES_CONFIRMED
3,Ethical Purchasing Group - SIA Project 22 23,Low-Level Goals,44,43,-1,0.5135,0.5254,0.0119,0.7531,0.7531,0.0000,0.6107,0.6190,0.0084,True,ALL_BRANCHES_CONFIRMED
4,Event Organization Portal - SIA Project 21 22,High-Level Goals,11,10,-1,0.2196,0.2415,0.0220,0.8051,0.8051,0.0000,0.3450,0.3716,0.0265,True,ALL_BRANCHES_CONFIRMED
5,Event Organization Portal - SIA Project 21 22,Low-Level Goals,61,56,-5,0.3997,0.4338,0.0341,0.8128,0.8098,-0.0029,0.5359,0.5650,0.0291,True,ALL_BRANCHES_CONFIRMED
6,Genome Nexus,High-Level Goals,10,8,-2,0.6620,0.8126,0.1506,0.7355,0.7223,-0.0132,0.6968,0.7648,0.0680,True,ALL_BRANCHES_CONFIRMED
7,Genome Nexus,Low-Level Goals,31,25,-6,0.8198,0.8243,0.0046,0.7474,0.6061,-0.1413,0.7819,0.6986,-0.0834,True,ALL_BRANCHES_CONFIRMED
8,Gestao Hospital,High-Level Goals,10,6,-4,0.3077,0.4989,0.1912,0.7692,0.7484,-0.0208,0.4396,0.5987,0.1592,False,NO_ACTIONS_REMAIN_WITH_WARNINGS
9,Gestao Hospital,Low-Level Goals,33,20,-13,0.4559,0.6985,0.2426,0.7522,0.6985,-0.0537,0.5677,0.6985,0.1308,False,NO_ACTIONS_REMAIN_WITH_WARNINGS



Rows where bottom-up improved F1:


,dataset,level,top_down_goal_count,bottom_up_goal_count,delta_goal_count,top_down_precision,bottom_up_precision,delta_precision,top_down_recall,bottom_up_recall,delta_recall,top_down_f1,bottom_up_f1,delta_f1,cycle_converged,cycle_stop_reason
8,Gestao Hospital,High-Level Goals,10,6,-4,0.3077,0.4989,0.1912,0.7692,0.7484,-0.0208,0.4396,0.5987,0.1592,False,NO_ACTIONS_REMAIN_WITH_WARNINGS
9,Gestao Hospital,Low-Level Goals,33,20,-13,0.4559,0.6985,0.2426,0.7522,0.6985,-0.0537,0.5677,0.6985,0.1308,False,NO_ACTIONS_REMAIN_WITH_WARNINGS
10,La Reine Marlene - SIA Project 23 24,High-Level Goals,24,19,-5,0.3323,0.4199,0.0875,0.7976,0.7977,0.0001,0.4692,0.5502,0.0810,False,NO_ACTIONS_REMAIN_WITH_WARNINGS
11,La Reine Marlene - SIA Project 23 24,Low-Level Goals,86,70,-16,0.4554,0.5582,0.1028,0.7833,0.7815,-0.0018,0.5759,0.6512,0.0753,False,NO_ACTIONS_REMAIN_WITH_WARNINGS
13,London Ambulance Service,Low-Level Goals,30,24,-6,0.2547,0.3189,0.0643,0.7640,0.7655,0.0015,0.3820,0.4503,0.0683,True,ALL_BRANCHES_CONFIRMED
6,Genome Nexus,High-Level Goals,10,8,-2,0.6620,0.8126,0.1506,0.7355,0.7223,-0.0132,0.6968,0.7648,0.0680,True,ALL_BRANCHES_CONFIRMED
15,SIA Project 25 26,Low-Level Goals,28,32,4,0.7313,0.7419,0.0107,0.5534,0.6417,0.0883,0.6300,0.6882,0.0582,True,ALL_BRANCHES_CONFIRMED
12,London Ambulance Service,High-Level Goals,10,8,-2,0.1531,0.1914,0.0383,0.7655,0.7655,0.0000,0.2552,0.3062,0.0510,True,ALL_BRANCHES_CONFIRMED
5,Event Organization Portal - SIA Project 21 22,Low-Level Goals,61,56,-5,0.3997,0.4338,0.0341,0.8128,0.8098,-0.0029,0.5359,0.5650,0.0291,True,ALL_BRANCHES_CONFIRMED
4,Event Organization Portal - SIA Project 21 22,High-Level Goals,11,10,-1,0.2196,0.2415,0.0220,0.8051,0.8051,0.0000,0.3450,0.3716,0.0265,True,ALL_BRANCHES_CONFIRMED



Rows where bottom-up worsened F1:


,dataset,level,top_down_goal_count,bottom_up_goal_count,delta_goal_count,top_down_precision,bottom_up_precision,delta_precision,top_down_recall,bottom_up_recall,delta_recall,top_down_f1,bottom_up_f1,delta_f1,cycle_converged,cycle_stop_reason
7,Genome Nexus,Low-Level Goals,31,25,-6,0.8198,0.8243,0.0046,0.7474,0.6061,-0.1413,0.7819,0.6986,-0.0834,True,ALL_BRANCHES_CONFIRMED
1,Assegno Unico Universale - SIA Project 24 25,Low-Level Goals,43,44,1,0.6482,0.6341,-0.0141,0.7963,0.7971,0.0008,0.7147,0.7063,-0.0083,True,ALL_BRANCHES_CONFIRMED


## 6. Riassunto macro tra dataset

La macro-media attribuisce lo stesso peso a ogni dataset. È utile come sintesi, mentre le righe per dataset restano necessarie per vedere dove il ciclo aiuta o danneggia la decomposizione.

In [8]:
macro_df = (
    metrics_df
    .groupby(["level", "method"], as_index=False)[["precision", "recall", "f1"]]
    .mean()
)

display(macro_df.round(4))

macro_pivot = macro_df.pivot(index="level", columns="method", values=["precision", "recall", "f1"])
print("\nMacro comparison:")
display(macro_pivot.round(4))

,level,method,precision,recall,f1
0,High-Level Goals,top_down,0.3270,0.7479,0.4323
1,High-Level Goals,top_down_plus_bottom_up,0.3914,0.7432,0.4823
2,Low-Level Goals,top_down,0.5348,0.7453,0.5998
3,Low-Level Goals,top_down_plus_bottom_up,0.5919,0.7317,0.6346



Macro comparison:


precision                           recall  \
method            top_down top_down_plus_bottom_up top_down   
level                                                         
High-Level Goals    0.3270                  0.3914   0.7479   
Low-Level Goals     0.5348                  0.5919   0.7453   

                                               f1                          
method           top_down_plus_bottom_up top_down top_down_plus_bottom_up  
level                                                                      
High-Level Goals                  0.7432   0.4323                  0.4823  
Low-Level Goals                   0.7317   0.5998                  0.6346

## 7. Export

I CSV seguenti sono già separati tra metriche assolute e delta A/B, così possono essere usati direttamente nelle tabelle della tesi.

In [9]:
metrics_path = EVALUATION_DIR / "top_down_vs_bottom_up_metrics.csv"
comparison_path = EVALUATION_DIR / "top_down_vs_bottom_up_deltas.csv"
macro_path = EVALUATION_DIR / "top_down_vs_bottom_up_macro_summary.csv"
summary_path = EVALUATION_DIR / "evaluation_summary.json"

metrics_df.to_csv(metrics_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
macro_df.to_csv(macro_path, index=False)

summary = {
    "datasets": common_datasets,
    "metric": {
        "embedding_model": BERT_MODEL_NAME,
        "matching": f"Hungarian one-to-one matching with cosine similarity >= {SIMILARITY_THRESHOLD}",
        "preprocessing": "Raw goal text; no stemming or lemmatization before BERT",
    },
    "outputs": {
        "metrics_csv": str(metrics_path),
        "deltas_csv": str(comparison_path),
        "macro_summary_csv": str(macro_path),
    },
}
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

for path in (metrics_path, comparison_path, macro_path, summary_path):
    print("Saved:", path)

Saved: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\top_down_vs_bottom_up\top_down_vs_bottom_up_metrics.csv
Saved: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\top_down_vs_bottom_up\top_down_vs_bottom_up_deltas.csv
Saved: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\top_down_vs_bottom_up\top_down_vs_bottom_up_macro_summary.csv
Saved: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\top_down_vs_bottom_up\evaluation_summary.json
